In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

dt = pd.read_csv('../data/processed/train_data.csv')

In [2]:
object_columns = dt.select_dtypes(include=['object']).columns

for col in object_columns:
    dt[col] = dt[col].str.lower().str.strip()

print("Log: Text standardization (lowercase & strip) completed for all object columns.")

Log: Text standardization (lowercase & strip) completed for all object columns.


In [3]:
hidden_missing_values = ['unknown', 'none', 'not defined', 'null', 'nan', '']
dt[object_columns] = dt[object_columns].replace(hidden_missing_values, np.nan)

print(f"Log: Hidden missing values {hidden_missing_values} converted to actual np.nan.")

Log: Hidden missing values ['unknown', 'none', 'not defined', 'null', 'nan', ''] converted to actual np.nan.


In [4]:
dt.describe(include='object').T

,count,unique,top,freq
ProductCD,590540,5,w,439670
card4,588963,4,visa,384767
card6,588969,4,debit,439938
P_emaildomain,496084,59,gmail.com,228355
R_emaildomain,137291,60,gmail.com,57147
M4,309096,3,m0,196405
id_23,5169,3,ip_proxy:transparent,3489
id_30,77565,75,windows 10,21155
id_31,140282,130,chrome 63.0,22000
id_33,73289,260,1920x1080,16874


In [7]:
#Identify numeric columns
numeric_cols = dt.select_dtypes(include=['int64', 'float64']).columns

#Find potential categorical columns based on unique value count
potential_categorical = []
threshold = 5 

for col in numeric_cols:
    unique_count = dt[col].nunique()
    
    #Exclude the target variable from this list to prevent data leakage
    if unique_count < threshold and col != 'isFraud':
        potential_categorical.append(col)

print(f"Found {len(potential_categorical)} disguised categorical columns.")

#Convert them to 'category' data type
dt[potential_categorical] = dt[potential_categorical].astype('category')

print("Type casting completed. Memory usage optimized.")

Found 0 disguised categorical columns.
Type casting completed. Memory usage optimized.


In [8]:
v_138_166 = [f'V{i}' for i in range(138, 167)]
v_322_339 = [f'V{i}' for i in range(322, 340)]
id_series = [f'id_{str(i).zfill(2)}' for i in [24, 25, 7, 8, 21, 26, 27, 23, 22, 18, 4, 3, 33, 9, 10, 30, 32, 34, 14]]

other_cols = ['dist2', 'D7', 'D13', 'D14', 'D12', 'D6', 'D8', 'D9','addr2', 'DeviceInfo', 'TransactionID', 'TransactionDT']
#branch-1
other_cols += ['P_emaildomain', 'R_emaildomain','M1', 'M2', 'M3', 'M4', 'M5', 'M7', 'M8', 'M9','id_12','id_15','id_16','id_23','id_27','id_28','id_29','id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38','DeviceType','DeviceInfo']

high_null_cols = v_138_166 + v_322_339 + id_series + other_cols

dt['TransactionAmt_Log'] = np.log1p(dt['TransactionAmt'])

dt1 = dt.drop(columns=list(high_null_cols) + ['TransactionAmt'])

In [9]:
missing_values = dt1.isnull().mean() * 100

# Drop all columns with missing values > 30%
THRESHOLD = 30
columns_to_drop = missing_values[missing_values > THRESHOLD].index.tolist()
dt1 = dt1.drop(columns=columns_to_drop)

In [10]:
categorical_columns = dt1.select_dtypes(include=['object', 'category']).columns
print(f"Log: Found {len(categorical_columns)} categorical columns for One-Hot Encoding.")

Log: Found 24 categorical columns for One-Hot Encoding.


In [11]:
for col in categorical_columns:
    dt1[col] = dt1[col].astype('object') 
    dt1[col] = dt1[col].fillna('missing_data')

print("Log: Missing values in categorical columns successfully imputed.")

dt1 = pd.get_dummies(dt1, columns=categorical_columns, drop_first=True, dtype=int)

print("Log: One-Hot Encoding applied successfully.")
print(f"Log: New dataset shape after OHE: {dt1.shape}")

Log: Missing values in categorical columns successfully imputed.
Log: One-Hot Encoding applied successfully.
Log: New dataset shape after OHE: (590540, 254)


In [12]:
dt1.head(5)

,isFraud,card1,card2,card3,card5,addr1,C1,C2,C3,C4,...,V121_1.0,V121_2.0,V121_3.0,V121_missing_data,V122_1.0,V122_2.0,V122_3.0,V122_missing_data,V305_2.0,V305_missing_data
0,0,13926,NaN,150.0,142.0,315.0,1.0,1.0,0.0,0.0,...,1,0,0,0,1,0,0,0,0,0
1,0,2755,404.0,150.0,102.0,325.0,1.0,1.0,0.0,0.0,...,1,0,0,0,1,0,0,0,0,0
2,0,4663,490.0,150.0,166.0,330.0,1.0,1.0,0.0,0.0,...,1,0,0,0,1,0,0,0,0,0
3,0,18132,567.0,150.0,117.0,476.0,2.0,5.0,0.0,0.0,...,1,0,0,0,1,0,0,0,0,0
4,0,4497,514.0,150.0,102.0,420.0,1.0,1.0,0.0,0.0,...,1,0,0,0,1,0,0,0,0,0


In [14]:
# --- MISSING VALUE ANALYSIS ---
print("--- MISSING VALUE ANALYSIS ---")

# Calculate missing value percentages for all columns
missing_values = dt1.isnull().sum()
missing_percentages = (missing_values / len(dt1)) * 100

# Create a DataFrame for better viewing
missing_df = pd.DataFrame({'Missing_Count': missing_values, 'Missing_Percentage': missing_percentages})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Percentage', ascending=False)

print(f"Total columns with missing values: {len(missing_df)} out of {dt.shape[1]}")
print("\nTop 15 columns with most missing values:")
display(missing_df.head(15))

--- MISSING VALUE ANALYSIS ---
Total columns with missing values: 157 out of 435

Top 15 columns with most missing values:


,Missing_Count,Missing_Percentage
V37,168969,28.612626
V36,168969,28.612626
V49,168969,28.612626
V48,168969,28.612626
V47,168969,28.612626
V46,168969,28.612626
V45,168969,28.612626
V44,168969,28.612626
V43,168969,28.612626
V42,168969,28.612626


In [15]:
dt1.to_csv('../data/processed/train_cleaned.csv', index=False)